In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

from src.resumo.cria_quadros import *

In [2]:
conta_grafica = r"data\input\anexo_b\Nova Base Diligencia 2014_v2.csv"

In [4]:
df = preparar_conta_grafica(conta_grafica)

In [3]:
from pathlib import Path
import numpy as np
import pandas as pd


COLUMNS = [
    "DATA GERACAO",
    "CONTADOR DEBITO",
    "NUM CONTRATO DEBITO",
    "CONTA DEBITO",
    "NOME CONTA DEBITO",
    "VALOR DEBITO",
    "CONTADOR CREDITO",
    "NUM CONTRATO CREDITO",
    "CONTA CREDITO",
    "NOME CONTA CREDITO",
    "VALOR CREDITO",
    "COSIF DEBITO",
    "NOME COSIF DEBITO",
    "COSIF CREDITO",
    "NOME COSIF CREDITO",
]

DEBITO = [
    "DATA GERACAO",
    "ANO",
    "CONTADOR DEBITO",
    "NUM CONTRATO DEBITO",
    "CONTA DEBITO",
    "NOME CONTA DEBITO",
    "VALOR DEBITO",
    "COSIF DEBITO",
    "NOME COSIF DEBITO",
]

CREDITO = [
    "DATA GERACAO",
    "ANO",
    "CONTADOR CREDITO",
    "NUM CONTRATO CREDITO",
    "CONTA CREDITO",
    "NOME CONTA CREDITO",
    "VALOR CREDITO",
    "COSIF CREDITO",
    "NOME COSIF CREDITO",
]

RENAME_DEBITO = {
    "DATA GERACAO": "Data",
    "ANO": "Ano",
    "CONTADOR DEBITO": "Contador",
    "NUM CONTRATO DEBITO": "Num Contrato",
    "CONTA DEBITO": "Conta",
    "NOME CONTA DEBITO": "Nome Conta",
    "VALOR DEBITO": "Valor Debito",
    "COSIF DEBITO": "COSIF",
    "NOME COSIF DEBITO": "Nome COSIF",
}

RENAME_CREDITO = {
    "DATA GERACAO": "Data",
    "ANO": "Ano",
    "CONTADOR CREDITO": "Contador",
    "NUM CONTRATO CREDITO": "Num Contrato",
    "CONTA CREDITO": "Conta",
    "NOME CONTA CREDITO": "Nome Conta",
    "VALOR CREDITO": "Valor Credito",
    "COSIF CREDITO": "COSIF",
    "NOME COSIF CREDITO": "Nome COSIF",
}


def tratar_valor(serie: pd.Series) -> pd.Series:
    return pd.to_numeric(
        serie.astype(str)
        .str.strip()
        .str.replace("+", "", regex=False)
        .str.replace(".", "", regex=False)
        .str.replace(",", ".", regex=False)
        .replace({"": None, "nan": None}),
        errors="coerce",
    )


def tratar_data(serie: pd.Series) -> pd.Series:
    limpa = (
        serie.astype(str)
        .str.strip()
        .replace({
            "99999999": None,
            "00000000": None,
            "": None,
            "nan": None,
        })
    )

    return pd.to_datetime(limpa, format="%Y%m%d", errors="coerce")


def montar_lado(
    df: pd.DataFrame,
    cols_origem: list[str],
    rename_map: dict[str, str],
    tipo: str,
) -> pd.DataFrame:
    out = df[cols_origem].copy().rename(columns=rename_map)

    if tipo == "C":
        out["Valor Debito"] = 0.0
        out["Tipo"] = "C"
    else:
        out["Valor Credito"] = 0.0
        out["Tipo"] = "D"

    return out


def processar_conta_grafica(conta_grafica: str | Path) -> pd.DataFrame:
    conta_grafica = Path(conta_grafica)

    df = pd.read_csv(
        conta_grafica,
        sep=";",
        encoding="utf-8",
        dtype=str,
    )

    if len(df.columns) < len(COLUMNS):
        raise ValueError(
            f"Arquivo com {len(df.columns)} colunas, esperado pelo menos {len(COLUMNS)}."
        )

    df = df.iloc[:, :len(COLUMNS)].copy()
    df.columns = COLUMNS

    df["VALOR DEBITO"] = tratar_valor(df["VALOR DEBITO"])
    df["VALOR CREDITO"] = tratar_valor(df["VALOR CREDITO"])
    df["DATA GERACAO"] = tratar_data(df["DATA GERACAO"])
    df["ANO"] = df["DATA GERACAO"].dt.strftime("%Y")

    df_credito = montar_lado(df, CREDITO, RENAME_CREDITO, tipo="C")
    df_debito = montar_lado(df, DEBITO, RENAME_DEBITO, tipo="D")

    df_final = pd.concat([df_credito, df_debito], ignore_index=True)

    cosif = df_final["COSIF"].fillna("").astype(str).str.strip()
    conta = df_final["Conta"].fillna("").astype(str).str.strip()
    nome_conta = df_final["Nome Conta"].fillna("").astype(str).str.strip()

    df_final["COSIF - Filtro"] = cosif.str[:5].replace("", np.nan)
    df_final["AnoMes"] = df_final["Data"].dt.strftime("%Y%m")
    df_final["Período de Apuração"] = np.where(
        df_final["Ano"].notna(),
        "31/12/" + df_final["Ano"],
        np.nan,
    )
    df_final["COSIF - Nivel 1"] = cosif.str[:3].str.ljust(14, "0").replace("", np.nan)
    df_final["COSIF - Nivel 2"] = cosif.str[:7].str.ljust(14, "0").replace("", np.nan)
    df_final["Conta + Descrição"] = np.where(
        conta.eq(""),
        np.nan,
        conta + " - " + nome_conta,
    )
    df_final["Valor Líquido"] = np.where(
        df_final["Tipo"] == "D",
        -df_final["Valor Debito"],
        df_final["Valor Credito"],
    )
    df_final["COSIF Apresentação"] = df_final["COSIF - Filtro"].str.replace(
        r"(\d)(\d)(\d)(\d{2}).*",
        r"\1.\2.\3.\4",
        regex=True,
    )

    return df_final